In [14]:
import pandas as pd
import plotly.graph_objects as go

# 1. Carregar os dados
df = pd.read_excel("tegument_less.xlsx")
# 2. Substituir valores vazios/NaN por 0 logo de cara
value_cols = [c for c in df.columns if c.startswith("SR") or c.startswith("RR") or c.startswith("NE")]
df[value_cols] = df[value_cols].fillna(0) 

# 2. Preencher células mescladas de Tissue e Protein
df[["Tissue", "Protein"]] = df[["Tissue", "Protein"]].ffill()

# 3. Transformar em formato longo (tidy) incluindo colunas extras do Excel
df_long = df.melt(
    id_vars=["Tissue", "Protein", "Peptides", "EpitopeNum", "EpitopeSeq"],
    value_vars=[c for c in df.columns if c.startswith("SR") or c.startswith("RR") or c.startswith("NE")],
    var_name="Sample",
    value_name="Value"
)


# 4. Definir ranges de proteína (agora para o eixo X)
protein_ranges = {
   "TSP-2 L.": (1, 78),
   "VAL8": (79, 196),
   "Carb.": (197, 286),
   "Annexin": (287, 401),
   "Annexin_1": (402, 513),
   "MEG-3.3": (514, 578),
   "MEG-3.2": (579, 640),
   "MEG-3.1": (641, 704),
   "LMWP": (705, 740)
}

x_ticks = [ (start + end) // 2 for start, end in protein_ranges.values() ]
x_labels = [label.replace(" ", "<br>") for label in protein_ranges.keys()]
for protein, (start, end) in protein_ranges.items():
    mid = (start + end) // 2
    x_ticks.append(mid)
    x_labels.append(protein)

In [15]:
# 5. Pegar o único tissue
tissue = df_long["Tissue"].unique()[0]
df_t = df_long[df_long["Tissue"] == tissue]
# 6. Construir o heatmap
heatmap = go.Heatmap(
    x=df_t["Peptides"],
    y=df_t["Sample"],
    z=df_t["Value"],
    colorscale=[
        [0.0, "rgb(255,255,255)"],   # branco
        [0.5, "rgb(0,176,240)"],     # azul
        [1.0, "rgb(255,0,0)"]        # vermelho
    ],
    colorbar=dict(title="MAPIX Score"),
    zmin=5000,   # mínimo da escala
    zmid=20000,  # ponto médio
    zmax=40000,  # máximo da escala
    hovertemplate=(
        "Sample: %{y}<br>"
        "Peptide: %{x}<br>"
        "Protein: %{customdata[0]}<br>"
        "Tissue: %{customdata[1]}<br>"
        "Value: %{z}<extra></extra>"
    ),
    customdata=df_t[["Protein","Tissue"]].values,
    xgap=0,
    ygap=0
)

# 7. Construir figura
fig = go.Figure(data=[heatmap])

# Adicionar divisores verticais entre proteínas
for _, (start, end) in protein_ranges.items():
    fig.add_vline(
        x=end + 0.5,
        line_width=1,
        line_dash="dot",
        line_color="black"
    )

In [16]:
# ---------- eixo Y: agora vamos mostrar os ticklabels originais coloridos ----------
y_tickvals = list(df_t["Sample"].unique())

# 🔧 ALTERAÇÃO 1: função para definir cor da fonte
def sample_fontcolor(sample):
    if str(sample).startswith("SR"):
        return "orangered"
    elif str(sample).startswith("RR"):
        return "green"
    elif str(sample).startswith("NE"):
        return "blue"
    else:
        return "black"


# adicionar anotações (uma por amostra) ao lado esquerdo do heatmap
y_ticktext = [
f"<span style='color:{sample_fontcolor(s)}'>{s}</span>"
for s in y_tickvals 
]


In [17]:
legenda_text = (
    "<b>Subtitles:</b><br>"
    "<span style='color:orange;'>SR</span> – Susceptible to Reinfection<br>"
    "<span style='color:green;'>RR</span> – Resistant to Reinfection<br>"
    "Carb - Carbonic anhydrase<br>"
    "TSP-2 L - Large Extracellular Loop of Tetraspanin-2"
    
)

fig.add_annotation(
    text=legenda_text,
    xref="paper", yref="paper",
    x=1.15, y=-0.5,   # ajusta a posição à direita
    showarrow=False,
    align="left",
    font=dict(size=12, color="black", family="Arial"),
    bordercolor="black",
    borderwidth=0
)
fig.update_yaxes(
    tickmode="array",
    tickvals=y_tickvals,
    ticktext=y_ticktext,
    showticklabels=True
 )
fig.update_layout(
    yaxis=dict(
        tickmode="array",
        tickvals=y_tickvals,
        #ticktext=x_ticktext,
        showticklabels=True,
        title_standoff=120  
    ),
    xaxis_title="<b>Peptides Microarray</b>",
    yaxis_title="<b>Samples</b>",
    xaxis=dict(
        tickmode="array",
        tickvals=x_ticks,
        ticktext=x_labels,
        showticklabels=True,
        tickangle=0,
        side="top"
    ),
    height=600,
    width=1300,
    dragmode="pan",
    plot_bgcolor="white",
    paper_bgcolor="white",
    autosize=True,
    margin=dict(b=180, t=80, l=100, r=250),
    coloraxis_colorbar=dict(
        title="MAPIX Score",
        tickvals=[5000, 20000, 40000],
        ticktext=["5k", "20k", "40k"]
    )
)

# ---------- Adicionar numeração automática dos epítopos ----------
df_epitopos = df_t.drop_duplicates(subset=["Peptides", "EpitopeNum", "EpitopeSeq"])
df_epitopos = df_epitopos[df_epitopos["EpitopeNum"].notna()]

for _, row in df_epitopos.iterrows():
    fig.add_annotation(
        x=row["Peptides"],
        y=-0.08,
        xref="x",
        yref="paper",
        text=str(int(row["EpitopeNum"])),
        showarrow=False,
        font=dict(color="black", size=12, family="Arial"),
        xanchor="center",
        yanchor="bottom",
        hovertext=f"Epitope {int(row['EpitopeNum'])}: {row['EpitopeSeq']}",
        hoverlabel=dict(bgcolor="white")
    )
    
fig.add_annotation(
    text="double-click to reset",
    xref="paper", yref="paper",
    x=0, y=-0.2,   # 🔽 posição abaixo do eixo X
    showarrow=False,
    font=dict(size=12, color="black", family="Arial"),
    align="left"
)
# Adicionar título "EPITOPES" abaixo do eixo X, à esquerda dos números
fig.add_annotation(
    text="<b>Epitopes<b>",
    xref="paper", yref="paper",
    x=-0.08, y=-0.03,          # posição relativa no espaço do gráfico
    showarrow=False,
    font=dict(size=13, color="black", family="Arial"),
    xanchor="left",
    yanchor="top",
    align="left"
)
# Exportar com scroll zoom ativado
fig.write_html(
    "heatmap_invertido_tegument_less.html",
    include_plotlyjs="cdn",
    full_html=True,
    default_width="100%",
    default_height="100%",
    config={"scrollZoom": True}
)